In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import os
import json
import time
import warnings
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    matthews_corrcoef
)

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

DRIVE_PATH = '/content/drive/MyDrive/CICIoT2023_Research'
os.makedirs(f'{DRIVE_PATH}/models_cat8_dl', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/results_cat8_dl', exist_ok=True)

CATEGORY_ENCODING = {
    'DDoS': 0,
    'DoS': 1,
    'Mirai': 2,
    'Benign': 3,
    'Recon': 4,
    'Spoofing': 5,
    'Web': 6,
    'BruteForce': 7
}
ID_TO_CATEGORY = {v: k for k, v in CATEGORY_ENCODING.items()}
target_names = [ID_TO_CATEGORY[i] for i in range(8)]

class ValF1Callback:
    """Compute val F1-macro after every epoch and append to self.val_f1."""

    def __init__(self, X_val, y_val, batch_size):
        self._base = keras.callbacks.Callback()
        self._base.on_epoch_end = self._on_epoch_end
        self.X_val = X_val
        self.y_val = y_val
        self.batch_size = batch_size
        self.val_f1 = []

    def _on_epoch_end(self, epoch, logs=None):
        prob = self._base.model.predict(self.X_val, batch_size=self.batch_size, verbose=0)
        pred = np.argmax(prob, axis=1)
        score = f1_score(self.y_val, pred, average='macro', zero_division=0)
        self.val_f1.append(float(score))
        if logs is not None:
            logs['val_f1_macro'] = score

    @property
    def callback(self):
        return self._base


Device: cuda
GPU: Tesla T4
PyTorch: 2.10.0+cu128


In [ ]:
print("Loading multiclass DL data...")

X_train = pd.read_csv(f'{DRIVE_PATH}/X_train_balanced_8cat_smoteenn.csv')
y_train = pd.read_csv(f'{DRIVE_PATH}/y_train_balanced_8cat_smoteenn.csv').squeeze()

X_val = pd.read_csv(f'{DRIVE_PATH}/X_val_cat8.csv')
y_val = pd.read_csv(f'{DRIVE_PATH}/y_val_cat8.csv').squeeze()

X_test = pd.read_csv(f'{DRIVE_PATH}/X_test_cat8.csv')
y_test = pd.read_csv(f'{DRIVE_PATH}/y_test_cat8.csv').squeeze()

X_train_np = X_train.values.astype(np.float32)
X_val_np = X_val.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)

y_train_np = y_train.values.astype(np.int32)
y_val_np = y_val.values.astype(np.int32)
y_test_np = y_test.values.astype(np.int32)

n_features = X_train_np.shape[1]
num_classes = 8

print("Train:", X_train_np.shape, y_train_np.shape)
print("Val:", X_val_np.shape, y_val_np.shape)
print("Test:", X_test_np.shape, y_test_np.shape)

Category encoding:
  0: DDoS
  1: DoS
  2: Mirai
  3: Recon
  4: Spoofing
  5: Web
  6: BruteForce
  7: Benign


In [ ]:
def compute_multiclass_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1_weighted": f1_score(y_true, y_pred, average='weighted'),
        "f1_macro": f1_score(y_true, y_pred, average='macro'),
        "mcc": matthews_corrcoef(y_true, y_pred)
    }

Loading X_val for feature names...
X_val: (2100516, 39)  (12.4s)
Loading balanced training data...
X_train: (1629654, 39)  (10.5s)
Loading val labels...
Loading test data...
X_test: (4201031, 39)  (27.3s)

All loaded.
  Train: 1,629,654 (balanced)
  Val:   2,100,516 (imbalanced)
  Test:  4,201,031 (imbalanced)


In [ ]:
mlp = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
], name='MLP_CAT8')

mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

checkpoint_mlp = keras.callbacks.ModelCheckpoint(
    filepath=f'{DRIVE_PATH}/models_cat8_dl/mlp_cat8_best.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

val_f1_mlp = ValF1Callback(X_val_np, y_val_np, 4096)
t0 = time.time()
history_mlp = mlp.fit(
    X_train_np, y_train_np,
    validation_data=(X_val_np, y_val_np),
    epochs=15,
    batch_size=4096,
    callbacks=[early_stop, checkpoint_mlp, val_f1_mlp.callback],
    verbose=1
)
mlp_time = time.time() - t0

Tensors created.
  X_train_t: torch.Size([1629654, 39])
  X_val_t:   torch.Size([2100516, 39])
  X_test_t:  torch.Size([4201031, 39])
  Batch size: 2048
  Train batches per epoch: 796


In [ ]:
X_train_cnn = X_train_np.reshape(-1, n_features, 1)
X_val_cnn = X_val_np.reshape(-1, n_features, 1)
X_test_cnn = X_test_np.reshape(-1, n_features, 1)

cnn = keras.Sequential([
    layers.Input(shape=(n_features, 1)),
    layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),

    layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation='softmax')
], name='CNN_CAT8')

cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop_cnn = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

checkpoint_cnn = keras.callbacks.ModelCheckpoint(
    filepath=f'{DRIVE_PATH}/models_cat8_dl/cnn_cat8_best.keras',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

val_f1_cnn = ValF1Callback(X_val_cnn, y_val_np, 4096)
t0 = time.time()
history_cnn = cnn.fit(
    X_train_cnn, y_train_np,
    validation_data=(X_val_cnn, y_val_np),
    epochs=15,
    batch_size=4096,
    callbacks=[early_stop_cnn, checkpoint_cnn, val_f1_cnn.callback],
    verbose=1
)
cnn_time = time.time() - t0

Helper functions defined.


In [ ]:
mlp_prob = mlp.predict(X_test_np, batch_size=4096, verbose=1)
mlp_pred = np.argmax(mlp_prob, axis=1)
mlp_metrics = compute_multiclass_metrics(y_test_np, mlp_pred)

print("MLP metrics:", mlp_metrics)
print(classification_report(y_test_np, mlp_pred, target_names=target_names, digits=4))

cnn_prob = cnn.predict(X_test_cnn, batch_size=4096, verbose=1)
cnn_pred = np.argmax(cnn_prob, axis=1)
cnn_metrics = compute_multiclass_metrics(y_test_np, cnn_pred)

print("CNN metrics:", cnn_metrics)
print(classification_report(y_test_np, cnn_pred, target_names=target_names, digits=4))
# ── Train metrics (balanced training set) ─────────────────────────────────
mlp_train_prob = mlp.predict(X_train_np, batch_size=4096, verbose=0)
mlp_train_pred = np.argmax(mlp_train_prob, axis=1)
mlp_train_metrics = compute_multiclass_metrics(y_train_np, mlp_train_pred)

print("=== MLP TRAIN metrics ===", mlp_train_metrics)

cnn_train_prob = cnn.predict(X_train_cnn, batch_size=4096, verbose=0)
cnn_train_pred = np.argmax(cnn_train_prob, axis=1)
cnn_train_metrics = compute_multiclass_metrics(y_train_np, cnn_train_pred)

print("=== CNN TRAIN metrics ===", cnn_train_metrics)


Training MLP for 30 epochs...
 Epoch   Train Loss   Val F1 Mac     Time
------------------------------------------
     1       0.2792       0.5920    38.7s ← best
     2       0.1948       0.6105    37.9s ← best
     3       0.1801       0.6066    37.4s
     4       0.1730       0.6115    36.9s ← best
     5       0.1682       0.6120    37.8s ← best
     6       0.1651       0.6053    38.5s
     7       0.1621       0.6088    38.2s
     8       0.1604       0.6086    37.9s
     9       0.1585       0.6118    38.1s
    10       0.1535       0.6113    38.1s
    11       0.1519       0.6171    38.2s ← best
    12       0.1509       0.6156    38.3s
    13       0.1500       0.6140    38.6s
    14       0.1487       0.6191    38.4s ← best
    15       0.1480       0.6203    38.4s ← best
    16       0.1476       0.6207    38.1s ← best
    17       0.1470       0.6200    37.9s
    18       0.1465       0.6225    37.8s ← best
    19       0.1462       0.6217    38.0s
    20       0.1456    

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, pred, title in zip(
    axes,
    [mlp_pred, cnn_pred],
    ['MLP', '1D CNN']
):
    cm = confusion_matrix(y_test_np, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=target_names)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', xticks_rotation=45)
    ax.set_title(f'{title} — Test Confusion Matrix', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/results_cat8_dl/cat8_dl_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

=== MLP VALIDATION ===

MLP — Validation
  Accuracy:      0.7863
  F1 (macro):    0.6290
  F1 (weighted): 0.8000

              precision    recall  f1-score   support

        DDoS       0.93      0.74      0.83   1228858
         DoS       0.53      0.83      0.64    417978
       Mirai       0.99      1.00      1.00    235768
       Recon       0.86      0.53      0.65     65758
    Spoofing       0.89      0.68      0.77     43696
         Web       0.06      0.58      0.10      2490
  BruteForce       0.15      0.46      0.23      1237
      Benign       0.78      0.85      0.81    104731

    accuracy                           0.79   2100516
   macro avg       0.65      0.71      0.63   2100516
weighted avg       0.84      0.79      0.80   2100516

=== MLP TEST (Imbalanced) ===

MLP — Test (Imbalanced)
  Accuracy:      0.7863
  F1 (macro):    0.6274
  F1 (weighted): 0.8001

              precision    recall  f1-score   support

        DDoS       0.93      0.74      0.83   245820

In [ ]:
dl_results = [
    {
        'Model': 'MLP',
        'Test Accuracy': mlp_metrics['accuracy'],
        'Test Balanced Accuracy': mlp_metrics['balanced_accuracy'],
        'Test F1 Weighted': mlp_metrics['f1_weighted'],
        'Test F1 Macro': mlp_metrics['f1_macro'],
        'Test MCC': mlp_metrics['mcc'],
        'Train Accuracy': mlp_train_metrics['accuracy'],
        'Train F1 Macro': mlp_train_metrics['f1_macro'],
        'Train F1 Weighted': mlp_train_metrics['f1_weighted'],
        'Train Time(s)': mlp_time
    },
    {
        'Model': '1D CNN',
        'Test Accuracy': cnn_metrics['accuracy'],
        'Test Balanced Accuracy': cnn_metrics['balanced_accuracy'],
        'Test F1 Weighted': cnn_metrics['f1_weighted'],
        'Test F1 Macro': cnn_metrics['f1_macro'],
        'Test MCC': cnn_metrics['mcc'],
        'Train Accuracy': cnn_train_metrics['accuracy'],
        'Train F1 Macro': cnn_train_metrics['f1_macro'],
        'Train F1 Weighted': cnn_train_metrics['f1_weighted'],
        'Train Time(s)': cnn_time
    }
]

dl_results_df = pd.DataFrame(dl_results)
dl_results_df.to_csv(f'{DRIVE_PATH}/results_cat8_dl/cat8_dl_results.csv', index=False)
print(dl_results_df)

In [ ]:
mlp.save(f'{DRIVE_PATH}/models_cat8_dl/mlp_cat8_final.keras')
cnn.save(f'{DRIVE_PATH}/models_cat8_dl/cnn_cat8_final.keras')

history_bundle = {
    'mlp_history': history_mlp.history,
    'cnn_history': history_cnn.history
}

with open(f'{DRIVE_PATH}/results_cat8_dl/cat8_dl_training_history.json', 'w') as f:
    json.dump(history_bundle, f, indent=2)

print("Saved DL models and training history.")
experiment_log = {
    'script': '08_Baseline_Multiclass_DL',
    'framework': 'tensorflow/keras',
    'dataset': 'CICIoT2023',
    'task': 'cat8_multiclass',
    'epochs': 15,
    'batch_size': 4096,
    'use_class_weights': False,
    'train_balanced': True,
    'test_imbalanced': True,
    'models': {
        'MLP': {
            'test_accuracy':          round(mlp_metrics['accuracy'], 4),
            'test_balanced_accuracy': round(mlp_metrics['balanced_accuracy'], 4),
            'test_f1_macro':          round(mlp_metrics['f1_macro'], 4),
            'test_f1_weighted':       round(mlp_metrics['f1_weighted'], 4),
            'test_mcc':               round(mlp_metrics['mcc'], 4),
            'train_accuracy':         round(mlp_train_metrics['accuracy'], 4),
            'train_f1_macro':         round(mlp_train_metrics['f1_macro'], 4),
            'train_f1_weighted':      round(mlp_train_metrics['f1_weighted'], 4),
            'train_time_s':           round(mlp_time, 1),
        },
        '1D CNN': {
            'test_accuracy':          round(cnn_metrics['accuracy'], 4),
            'test_balanced_accuracy': round(cnn_metrics['balanced_accuracy'], 4),
            'test_f1_macro':          round(cnn_metrics['f1_macro'], 4),
            'test_f1_weighted':       round(cnn_metrics['f1_weighted'], 4),
            'test_mcc':               round(cnn_metrics['mcc'], 4),
            'train_accuracy':         round(cnn_train_metrics['accuracy'], 4),
            'train_f1_macro':         round(cnn_train_metrics['f1_macro'], 4),
            'train_f1_weighted':      round(cnn_train_metrics['f1_weighted'], 4),
            'train_time_s':           round(cnn_time, 1),
        },
    }
}

with open(f'{DRIVE_PATH}/results_cat8_dl/experiment_log_multiclass_dl.json', 'w') as f:
    json.dump(experiment_log, f, indent=2)

print("Saved experiment log.")